# 16.72 - Train-only screened-data adaptation for tooth spacing

## Question and fixed comparison

Does a predeclared image/task-compatibility screen on an external train-only source improve the spacing detector when the architecture, tooth-anatomy initialization, NWD loss, seed, training schedule, and locked validation are held constant?

| Arm | Training data | Loss and model |
|---|---|---|
| Control | Locked project train split | YOLO26s + common tooth-anatomy initialization + CIoU-NWD |
| Screened-data adaptation | Same project train split plus 93 screened external train images repeated four times | Identical |

The 93-image candidate has 165 boxes. It excludes 26 predeclared composite/out-of-scope images from the 119-image source train partition; 84 retained examples remain in a manual-review category and are not clinically adjudicated. The 22 positive-only source-validation examples and all source-test data are absent from the package and are never opened. The main comparison uses the same 477-image project validation and its 473-image overlap-excluded sensitivity view. We do not choose validation images to raise scores.

This is a data-quality ablation, not a claim of clinical generalization. The project validation has already guided development and is not an independent test.

## 1. Pin runtime and verify the two Kaggle T4 devices

In [ ]:
%pip -q install ultralytics==8.4.183

In [ ]:
from pathlib import Path
import hashlib, json, math, os, random, re, shutil, subprocess, sys, time
import numpy as np, pandas as pd, torch, yaml
from PIL import Image
from IPython.display import display
from ultralytics import YOLO

INPUT=Path('/kaggle/input'); OUT=Path('/kaggle/working/spacing_addon'); OUT.mkdir(parents=True,exist_ok=True); TMP=Path('/tmp/spacing_addon'); TMP.mkdir(parents=True,exist_ok=True)
assert torch.cuda.is_available() and torch.cuda.device_count()>=2
GPUS=[torch.cuda.get_device_name(i) for i in range(2)]; assert all('T4' in n for n in GPUS),GPUS
assert __import__('ultralytics').__version__=='8.4.183'
print({'gpus':GPUS,'torch':torch.__version__,'ultralytics':__import__('ultralytics').__version__})

## 2. Verify both immutable manifests and keep the source test unopened

In [ ]:
EXPECTED_MANIFEST='38cba07194707131b629b020ea9078d487dd397e13221d88075ca8d9cf7a3e9b'
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
manifest_paths=[p for p in INPUT.rglob('manifest.csv') if sha256_file(p)==EXPECTED_MANIFEST]
assert len(manifest_paths)==1,manifest_paths
SOURCE=manifest_paths[0].parent; manifest=pd.read_csv(manifest_paths[0],dtype={'sample_id':str})
assert len(manifest)==15511 and manifest.sample_id.is_unique
assert manifest.split.value_counts().to_dict()=={'train':15034,'valid':477}
assert not (SOURCE/'test').exists()
TRAIN_OVERLAP_IDS={'d7f3e2f640c5c219cf0486d4','cdd4842bb15ff4a10286b1d2','2d317bf73290741503a7b773','84558726296547622c33c6cf'}
VALID_OVERLAP_IDS={'ebb582d897abd9e1f55f4618','1a279792c0f591cf6dabe546','f8c55f96a0c5d7e0fe735d32','5b52ca77ecd22069f79c356d'}
assert TRAIN_OVERLAP_IDS.issubset(set(manifest.loc[manifest.split.eq('train'),'sample_id']))
assert VALID_OVERLAP_IDS.issubset(set(manifest.loc[manifest.split.eq('valid'),'sample_id']))
train_rows=manifest[manifest.split.eq('train')&~manifest.sample_id.isin(TRAIN_OVERLAP_IDS)].copy()
valid_rows=manifest[manifest.split.eq('valid')].copy()
sensitivity_rows=valid_rows[~valid_rows.sample_id.isin(VALID_OVERLAP_IDS)].copy()
assert (len(train_rows),len(valid_rows),len(sensitivity_rows))==(15030,477,473)
print({'manifest_sha256':EXPECTED_MANIFEST,'train_images':len(train_rows),'primary_valid':len(valid_rows),'sensitivity_valid':len(sensitivity_rows),'source_test_opened':False})

ADDON_MANIFEST_SHA='672caf0182dd9b2479dcc3f1ef0ec7cc956bff0e02f68cd0a8780612020bc0bc'
addon_manifests=[p for p in INPUT.rglob('manifest.csv') if sha256_file(p)==ADDON_MANIFEST_SHA]
assert len(addon_manifests)==1, addon_manifests
ADDON=addon_manifests[0].parent
addon_manifest=pd.read_csv(addon_manifests[0])
assert len(addon_manifest)==93 and addon_manifest.output_image.is_unique
assert int(addon_manifest.boxes.sum())==165
assert set(addon_manifest.visual_triage).issubset({'manual_review','retain_candidate'})
assert not (ADDON/'valid').exists() and not (ADDON/'test').exists()
receipt=json.loads((ADDON/'package_receipt.json').read_text(encoding='utf-8'))
assert receipt['source_validation_included'] is False and receipt['source_test_included'] is False
addon_train_rows=addon_manifest.copy()
print({'addon_manifest_sha256':ADDON_MANIFEST_SHA,'addon_train_images':len(addon_train_rows),'addon_train_boxes':int(addon_train_rows.boxes.sum()),'source_validation_included':False,'source_test_opened':False})

## 3. Build matched base and 4x screened add-on training manifests

Only train images are modified between arms. The locked project validation and sensitivity IDs are copied unchanged into both YAML configurations. The 93 screened external images are repeated four times in the add-on arm. No external validation or test data are attached. Prepared files go under `/tmp`, so Kaggle output contains model evidence rather than a second copy of the dataset.

In [ ]:
BASE=TMP/'base'; ADDON_RUN=TMP/'addon4x'; counts={}
def prepare_project_split(root, split, rows):
    image_dir=root/split/'images'; label_dir=root/split/'labels'
    image_dir.mkdir(parents=True,exist_ok=True); label_dir.mkdir(parents=True,exist_ok=True)
    boxes=positives=0
    for row in rows.itertuples(index=False):
        image=SOURCE/row.image; label=SOURCE/row.label
        assert image.is_file() and label.is_file()
        raw=label.read_bytes(); assert hashlib.sha256(raw).hexdigest()==row.label_sha256
        kept=[]
        for line in raw.decode('utf-8').splitlines():
            fields=line.split(); assert len(fields)==5 and fields[0] in ('0','1')
            if fields[0]=='0': kept.append(line)
        assert len(kept)==int(row.gap_or_spacing_boxes)
        target=image_dir/image.name
        if not target.exists(): target.symlink_to(image)
        (label_dir/(image.stem+'.txt')).write_text(('\n'.join(kept)+'\n') if kept else '',encoding='utf-8')
        boxes+=len(kept); positives+=bool(kept)
    return {'images':len(rows),'positive_images':int(positives),'boxes':int(boxes)}
for root in [BASE,ADDON_RUN]:
    counts[str(root.name)]={}
    for split,rows in [('train',train_rows),('valid',valid_rows),('valid_sensitivity',sensitivity_rows)]:
        counts[str(root.name)][split]=prepare_project_split(root,split,rows)
base_train_images=BASE/'train'/'images'; base_train_labels=BASE/'train'/'labels'
addon_img_dir=ADDON_RUN/'train'/'images'; addon_lbl_dir=ADDON_RUN/'train'/'labels'
for repeat in range(4):
    for row in addon_train_rows.itertuples(index=False):
        image=ADDON/'train'/'images'/row.output_image
        label=ADDON/'train'/'labels'/(Path(row.output_image).stem+'.txt')
        assert image.is_file() and label.is_file()
        assert hashlib.sha256(image.read_bytes()).hexdigest()==row.image_sha256
        assert hashlib.sha256(label.read_bytes()).hexdigest()==row.label_sha256
        name=f'ext{repeat}_{row.output_image}'
        (addon_img_dir/name).symlink_to(image)
        shutil.copyfile(label,addon_lbl_dir/(Path(name).stem+'.txt'))
addon_train_count=len(list(addon_img_dir.iterdir()))
addon_box_count=sum(len(p.read_text().splitlines()) for p in addon_lbl_dir.glob('*.txt'))
assert addon_train_count==15030+4*93 and addon_box_count==8223+4*165
for root in [BASE,ADDON_RUN]:
    (root/'data.yaml').write_text(yaml.safe_dump({'path':str(root),'train':'train/images','val':'valid/images','names':{0:'gap_or_spacing'}},sort_keys=False))
    (root/'sensitivity.yaml').write_text(yaml.safe_dump({'path':str(root),'train':'train/images','val':'valid_sensitivity/images','names':{0:'gap_or_spacing'}},sort_keys=False))
assert counts['base']['valid']==counts['addon4x']['valid'] and counts['base']['valid_sensitivity']==counts['addon4x']['valid_sensitivity']
display(pd.DataFrame({'control':counts['base'],'screened_93_4x':counts['addon4x']}))
print({'addon_arm_train_images':addon_train_count,'addon_arm_boxes':addon_box_count,'external_validation_included':False})

## 4. Reconstruct the same common tooth-anatomy initialization for both arms

In [ ]:
EXPECTED_MASK_CKPT='e70739ddc2d48163d2aab75bc4116294189253c325e77f960dbe02aad4dd5337'
mask_matches=[p for p in INPUT.rglob('best.pt') if sha256_file(p)==EXPECTED_MASK_CKPT]
assert len(mask_matches)==1,mask_matches
mask_model=YOLO(str(mask_matches[0])); detector=YOLO('yolo26s.pt')
src=mask_model.model.state_dict(); dst=detector.model.state_dict(); detect_head=len(detector.model.model)-1
def layer_number(key):
    m=re.match(r'model\.(\d+)\.',key); return int(m.group(1)) if m else None
shared={k:v for k,v in src.items() if k in dst and tuple(v.shape)==tuple(dst[k].shape) and layer_number(k) is not None and layer_number(k)<detect_head}
target={k:v for k,v in dst.items() if layer_number(k) is not None and layer_number(k)<detect_head}
transfer_fraction=sum(v.numel() for v in shared.values())/sum(v.numel() for v in target.values())
assert len(shared)==468 and transfer_fraction>=.90
detector.model.load_state_dict(shared,strict=False)
COMMON_INIT=OUT/'common_anatomy_init.pt'; detector.save(COMMON_INIT)
assert COMMON_INIT.is_file()
print({'mask_checkpoint_sha256':EXPECTED_MASK_CKPT,'common_init_sha256':sha256_file(COMMON_INIT),'copied_tensors':len(shared),'representation_parameter_fraction':transfer_fraction,'task_head_copied':False})

## 5. Compute the NWD scale from the locked project train split only

The NWD normalizer remains unchanged between arms: only the original project train boxes contribute. This keeps the loss scale controlled and avoids allowing external or project validation labels to influence training configuration.

In [ ]:
train_sizes=[]
for row in train_rows.itertuples(index=False):
    raw=(SOURCE/row.label).read_text(encoding='utf-8')
    with Image.open(SOURCE/row.image) as im: width,height=im.size
    resize_scale=min(640/width,640/height)
    for line in raw.splitlines():
        if not line.strip():continue
        cls,xc,yc,bw,bh=line.split()
        if cls=='0': train_sizes.append((float(bw)*width*resize_scale,float(bh)*height*resize_scale))
sizes=np.asarray(train_sizes); C_NWD=float(np.median(np.sqrt(sizes[:,0]*sizes[:,1])))
assert len(sizes)==8223 and 5.0<=C_NWD<=80.0
NWD_LAMBDA=.5
print({'train_target_count':len(sizes),'median_width_640':float(np.median(sizes[:,0])),
       'median_height_640':float(np.median(sizes[:,1])),'NWD_C_pixels':C_NWD,
       'NWD_mix_weight':NWD_LAMBDA,'validation_used_for_C':False})

## 6. Define one shared NWD worker

Both arms use CIoU-NWD. They differ only in training-data membership and the declared 4x repetition of the screened external train partition. Both use the same seed, initialization hash, epochs, optimizer, augmentations, image size, batch size and validation IDs.

In [ ]:
worker_source = r'''import argparse,hashlib,json,time
from pathlib import Path
import pandas as pd, torch
from ultralytics import YOLO
from ultralytics.utils.loss import BboxLoss

def patch_nwd(C, mix):
    original=BboxLoss.forward
    def forward(self,pred_dist,pred_bboxes,anchor_points,target_bboxes,target_scores,target_scores_sum,fg_mask,imgsz,stride):
        ciou_loss,l1_loss=original(self,pred_dist,pred_bboxes,anchor_points,target_bboxes,target_scores,target_scores_sum,fg_mask,imgsz,stride)
        selected=fg_mask.nonzero(as_tuple=True)
        if len(selected[0])==0:return ciou_loss,l1_loss
        weight=target_scores[selected].sum(-1,keepdim=True)
        scale=stride[selected[1]]
        pred=pred_bboxes[selected]*scale
        target=target_bboxes[selected]*scale
        pxy=(pred[:,:2]+pred[:,2:])*.5; txy=(target[:,:2]+target[:,2:])*.5
        pwh=(pred[:,2:]-pred[:,:2]).clamp(min=0); twh=(target[:,2:]-target[:,:2]).clamp(min=0)
        w2=((pxy-txy)**2).sum(-1,keepdim=True)+.25*((pwh-twh)**2).sum(-1,keepdim=True)
        distance=torch.sqrt(w2+1e-9)
        nwd_loss=((1-torch.exp(-distance/C))*weight).sum()/target_scores_sum
        return (1-mix)*ciou_loss+mix*nwd_loss,l1_loss
    BboxLoss.forward=forward

def sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
 return h.hexdigest()
p=argparse.ArgumentParser();p.add_argument('--arm',required=True);p.add_argument('--device',type=int,required=True);p.add_argument('--data',required=True);p.add_argument('--init',required=True);p.add_argument('--out',required=True);p.add_argument('--C',type=float,required=True);p.add_argument('--mix',type=float,required=True);a=p.parse_args()
patch_nwd(a.C,a.mix)
model=YOLO(a.init);root=Path(a.out);root.mkdir(parents=True,exist_ok=True);start=time.monotonic()
def guard(trainer):
 if time.monotonic()-start>5*3600:trainer.stop=True;print('Five-hour epoch-boundary guard reached',flush=True)
model.add_callback('on_train_epoch_end',guard)
model.train(data=a.data,epochs=60,patience=10,imgsz=640,batch=16,device=a.device,workers=2,seed=42,deterministic=True,optimizer='AdamW',lr0=.001,weight_decay=.0005,mosaic=.5,close_mosaic=10,project=str(root),name=a.arm,save=True,save_period=5,plots=True,verbose=True)
run=root/a.arm;best=run/'weights'/'best.pt';assert best.is_file(),best
history=pd.read_csv(run/'results.csv');history.columns=history.columns.str.strip()
val=YOLO(str(best)).val(data=a.data,imgsz=640,batch=16,device=a.device,workers=2,plots=True)
result={'arm':a.arm,'device':torch.cuda.get_device_name(a.device),'epochs_completed':len(history),'best_sha256':sha(best),'common_init_sha256':sha(a.init),'validation_reused':True,'source_test_used':False,'metrics':val.results_dict}
(run/'result.json').write_text(json.dumps(result,indent=2,default=str));print(json.dumps(result,indent=2,default=str),flush=True)
'''
worker_path=OUT/'worker.py';worker_path.write_text(worker_source,encoding='utf-8')
compile(worker_source,'spacing_nwd_worker','exec')
print({'worker_source_chars':len(worker_source),'method':'CIoU-NWD regression mixture','assignment':'native task-aligned IoU unchanged','L1_term':'unchanged'})

## 7. Launch the matched data comparison on separate T4 devices

In [ ]:
processes={}; logs={}; handles={}
arms_to_run=[('project_control',BASE/'data.yaml',0),('external_train_4x',ADDON_RUN/'data.yaml',1)]
for arm,data_yaml,gpu in arms_to_run:
    log_path=OUT/f'{arm}.log'; handle=log_path.open('w',encoding='utf-8'); handles[arm]=handle
    cmd=[sys.executable,str(worker_path),'--arm',arm,'--device',str(gpu),'--data',str(data_yaml),
         '--init',str(COMMON_INIT),'--out',str(OUT/arm),'--C',str(C_NWD),'--mix',str(NWD_LAMBDA)]
    processes[arm]=subprocess.Popen(cmd,stdout=handle,stderr=subprocess.STDOUT)
    print({'arm':arm,'gpu':gpu,'pid':processes[arm].pid,'log':str(log_path)},flush=True)
deadline=time.monotonic()+6.25*3600
while any(p.poll() is None for p in processes.values()):
    states={k:p.poll() for k,p in processes.items()}
    print({'worker_exit_codes':states,'elapsed_hours':round((6.25*3600-(deadline-time.monotonic()))/3600,2)},flush=True)
    if time.monotonic()>deadline:
        for p in processes.values():
            if p.poll() is None:p.terminate()
        raise TimeoutError('Safety guard reached. Preserve checkpoints and logs for continuation.')
    time.sleep(300)
for handle in handles.values():handle.close()
assert all(p.returncode==0 for p in processes.values()),{k:p.returncode for k,p in processes.items()}

## 8. Evaluate both checkpoints on the same fixed project cohorts

The external positive-only validation is intentionally not used to compute AP or the primary comparison. Primary validation is 477 images; sensitivity is the predeclared 473-image view with known overlap counterparts removed.

In [ ]:
arms={}
for arm,data_root,gpu in [('project_control',BASE,0),('external_train_4x',ADDON_RUN,1)]:
    run=OUT/arm/arm; best=run/'weights'/'best.pt'; assert best.is_file()
    history=pd.read_csv(run/'results.csv'); history.columns=history.columns.str.strip()
    primary=YOLO(str(best)).val(data=str(data_root/'data.yaml'),imgsz=640,batch=16,device=gpu,workers=2,plots=True)
    sensitivity=YOLO(str(best)).val(data=str(data_root/'sensitivity.yaml'),imgsz=640,batch=16,device=gpu,workers=2,plots=False)
    arms[arm]={'epochs':len(history),'checkpoint_sha256':sha256_file(best),'primary_477':primary.results_dict,'sensitivity_473':sensitivity.results_dict}
    shutil.copy2(best,OUT/f'{arm}_best.pt'); shutil.copy2(run/'results.csv',OUT/f'{arm}_results.csv')
summary={'experiment':'fixed_nwd_screened_external_train_addon_4x','spacing_manifest_sha256':EXPECTED_MANIFEST,
 'addon_manifest_sha256':ADDON_MANIFEST_SHA,'mask_pretrain_checkpoint_sha256':EXPECTED_MASK_CKPT,
 'common_initialization_sha256':sha256_file(COMMON_INIT),'project_train_images':15030,'project_train_boxes':8223,
 'addon_unique_train_images':93,'addon_unique_train_boxes':165,'addon_repeats':4,
 'addon_arm_train_images':addon_train_count,'addon_arm_train_boxes':addon_box_count,'NWD_C_input_pixels':C_NWD,
 'NWD_mix_weight':NWD_LAMBDA,'primary_validation_images':477,'sensitivity_validation_images':473,
 'external_validation_included':False,'source_test_opened':False,'validation_reused':True,
 'patient_independence_verified':False,'arms':arms}
(OUT/'comparison.json').write_text(json.dumps(summary,indent=2,default=str),encoding='utf-8')
display(pd.DataFrame({k:{'AP50':v['primary_477'].get('metrics/mAP50(B)'),'AP50-95':v['primary_477'].get('metrics/mAP50-95(B)'),'precision':v['primary_477'].get('metrics/precision(B)'),'recall':v['primary_477'].get('metrics/recall(B)'),'epochs':v['epochs']} for k,v in arms.items()}).T)
print(json.dumps(summary,indent=2,default=str))

## Interpretation and promotion rule

Report the paired primary and sensitivity results, checkpoint hashes, data counts and source attribution. Promote the screened-data arm only if it improves primary AP50 by at least 1 percentage point, does not reduce AP50-95 on either fixed project cohort, and does not reduce recall by more than 1.5 points. Otherwise retain the control and document the negative result. Do not select a validation subset or tune the reported cohort after seeing the results.

The screened source remains weakly supervised: 84 of its 93 images are still in manual-review status. This experiment does not establish patient independence or clinical performance.